In [1]:
## Step 1: Target Distribution

import numpy as np
import math

# Discrete Distribution

class MockSobol:
    """Simulates a QMCPy DiscreteDistribution (e.g., Sobol sequence) in s dimensions."""
    def __init__(self, dimension):
        self.dimension = dimension
        print(f"Driver initialized for dimension s={dimension} (target d={dimension - 1}).")

    def gen_points(self, M):
        """Generates M quasi-random points (using numpy.random.rand as a stand-in)."""
        if not math.log2(M).is_integer():
             print("Warning: M is not a power of 2. QMC properties may be degraded.")
        
        # In a real QMCPy package, this is an optimized low-discrepancy generator.
        return np.random.rand(M, self.dimension) 

# Target Density 

def example_target_density(x_coord):
    """Density: psi(x_1, x_2) = 2*x_1 for x_i in [0,1]^2"""
    # x_coord is a d-dimensional vector. We use only the first coordinate.
    return 2.0 * x_coord[0]

In [3]:
## Step 2: Defining the class with required parameters

class AcceptanceRejectionMeasure:
    """
    Implements the Deterministic Acceptance-Rejection (DAR) sampler 
    for QMC point sets on the Unit Cube [0, 1]^d.
    (Based on Zhu & Dick, 2014, Algorithm 2)
    """
    def __init__(self, target_density, discrete_dist, L, target_dim):
        # 1. Store Inputs
        self.target_density = target_density 
        self.discrete_dist = discrete_dist
        self.L = L 
        self.target_dim = target_dim 
        
        # 2. Check Dimensions: Driver dim (s) MUST be Target dim (d) + 1
        if self.discrete_dist.dimension != target_dim + 1:
            raise ValueError(
                f"Driver dim ({self.discrete_dist.dimension}) must be 1 + target dim ({target_dim})."
            )
        
        # 3. Acceptance Rate (C/L) - Assumed 0.5 for the example density psi(x)=2x
        self.expected_acceptance_rate = 0.5 

    def get_samples(self, N_target):
        """
        Executes the DAR sampling process to generate N_target accepted points.
        """
        
        # --- Step 1A: Calculate Total Driver Points (M) ---
        M_required = math.ceil(N_target / self.expected_acceptance_rate)
        M_required = int(M_required)

        # --- Step 1B: Generate QMC Points (s=d+1 dimension) ---
        print(f"\nRequesting M={M_required} points from {self.discrete_dist.dimension}-D QMC driver...")
        Q_m_s = self.discrete_dist.gen_points(M_required)  # M x s array
        
        accepted_points = []
        
        # --- Step 2: Apply DAR Filter ---
        print("Applying Acceptance-Rejection filter...")
        
        for x_full in Q_m_s:
            # 1. Driver Transformation: x = candidate (d dims), u = check variable (1 dim)
            x_candidate = x_full[:self.target_dim] 
            u_check = x_full[-1]          
            
            # 2. Evaluate Target Density
            psi_val = self.target_density(x_candidate)
            
            # 3. Acceptance Check: u <= psi(x) / L  (or L * u <= psi(x))
            if self.L * u_check <= psi_val:
                # If yes -> accept the d-dimensional candidate
                accepted_points.append(x_candidate)
            
            # Stop when we hit the target N
            if len(accepted_points) >= N_target:
                break
        
        # --- Step 3: Return a clean sample array ---
        print(f"Finished sampling. Accepted N={len(accepted_points)} points.")
        # Returns an N x d array
        return np.array(accepted_points)

In [4]:
## Parameters Setup for our Example

TARGET_DIMENSION = 2 # d=2 (x and y)
N_TARGET = 1024      # N, the desired number of accepted samples
L_BOUND = 2.0        # L, the max value of psi(x,y)=2x

# 1. Initialize the QMC Driver (s=3)
driver_seq = MockSobol(dimension=TARGET_DIMENSION + 1)

# 2. Initialize the DAR Measure
dar_measure = AcceptanceRejectionMeasure(
    target_density=example_target_density,
    discrete_dist=driver_seq,
    L=L_BOUND,
    target_dim=TARGET_DIMENSION
)

# 3. Generate the Samples
# The 'samples' array is the final P_N^(d) point set.
samples = dar_measure.get_samples(N_TARGET)

Driver initialized for dimension s=3 (target d=2).

Requesting M=2048 points from 3-D QMC driver...
Applying Acceptance-Rejection filter...
Finished sampling. Accepted N=987 points.


In [5]:
## Step 4: Verification ---
print("\n--- Verification (Simulated Step 4: Quality Check) ---")
print(f"Shape of final accepted sample array: {samples.shape}")

# Show a snippet of the results
print("\nFirst 5 accepted samples (points should be denser toward x1=1.0):")
print(samples[:5])

# Calculate the mean of the x_1 coordinate (the first column)
mean_x_coord = np.mean(samples[:, 0])
print(f"\nStatistical Check (Mean of x_1):")
print(f"  Expected Mean (for psi=2x): 0.6667")
print(f"  Calculated Sample Mean: {mean_x_coord:.4f}")

# (In a true QMCPy analysis, specialized functions would calculate the 
# star discrepancy over a delta-cover grid to empirically verify the 
# N^(-1/s) convergence rate shown in the paper.)


--- Verification (Simulated Step 4: Quality Check) ---
Shape of final accepted sample array: (987, 2)

First 5 accepted samples (points should be denser toward x1=1.0):
[[0.71977752 0.91874151]
 [0.92717873 0.75706286]
 [0.18326489 0.39446414]
 [0.39119816 0.91694576]
 [0.34138732 0.41671364]]

Statistical Check (Mean of x_1):
  Expected Mean (for psi=2x): 0.6667
  Calculated Sample Mean: 0.6633


In [2]:
## Using QMCPy package 

import numpy as np
import math
from typing import Union

# --- MOCK QMCPY DEPENDENCIES (ASSUMED TO BE AVAILABLE VIA IMPORTS) ---

class MethodImplementationError(Exception): pass
class ParameterError(Exception): pass
class AbstractDiscreteDistribution(object): pass
class AbstractTrueMeasure(object):
    """Minimal definition to satisfy inheritance requirements."""
    def __init__(self, d):
        self.d = d
        self.domain = np.tile([0, 1], (d, 1))
        self.range = np.tile([0, 1], (d, 1))

    # We must define these methods as they exist in the parent class structure, 
    # even if we override the top-level sampling.
    def _transform(self, x): raise MethodImplementationError(self, '_transform')
    def _weight(self, x): raise MethodImplementationError(self, '_weight')
    
    # Placeholder for the actual QMCPy __call__ and gen_samples methods 
    # which we will now fully override.
    def __call__(self, n=None, n_min=None, n_max=None, return_weights=False, warn=True):
        return self.gen_samples(n=n,n_min=n_min,n_max=n_max,return_weights=return_weights,warn=warn)
    def gen_samples(self, n=None, n_min=None, n_max=None, return_weights=False, warn=True):
        raise MethodImplementationError(self, 'gen_samples')

# --- MOCK SOBOL FOR DRIVER ---

class MockSobol(AbstractDiscreteDistribution):
    """Simulates a QMCPy DiscreteDistribution (e.g., Sobol sequence) in s dimensions."""
    def __init__(self, dimension):
        self.d = dimension  # Renamed 'dimension' to 'd' to match AbstractTrueMeasure/AbstractDiscreteDistribution
        self.mimics = 'StdUniform'

    def __call__(self, n=None, n_min=None, n_max=None, warn=True):
        """Simulates fetching the driver points M in s=d+1 dimensions."""
        M = n if n is not None else n_max - n_min
        return np.random.rand(M, self.d) 

# --- CORE QMCPy DAR IMPLEMENTATION: THE NEW TRUE MEASURE CLASS ---

class AcceptanceRejectionMeasure(AbstractTrueMeasure):
    """
    Implements the Deterministic Acceptance-Rejection (DAR) sampler (Zhu & Dick, 2014, Algorithm 2)
    as a TrueMeasure for non-uniform sampling on the unit cube [0, 1]^d.
    """
    
    def __init__(self, target_density, discrete_dist, L):
        
        # We must manually enforce the dimension relationship: s = d + 1
        target_dim = discrete_dist.d - 1 # Target dimension 'd'
        
        # 1. Initialize parent (AbstractTrueMeasure) with the *target* dimension 'd'
        super(AcceptanceRejectionMeasure, self).__init__(target_dim)
        
        # 2. Store DAR-specific properties
        self.target_density = target_density 
        self.L = L 
        self.driver_distrib = discrete_dist # The s=(d+1) dimensional driver
        self.driver_d = discrete_dist.d
        
        # 3. Define domain/range (required by AbstractTrueMeasure)
        # The output is in the unit cube [0,1]^d, so domain/range are both [0,1]^d
        self.domain = np.tile([0, 1], (target_dim, 1))
        self.range = np.tile([0, 1], (target_dim, 1))
        self.parameters = ['target_density', 'L'] # For __repr__
        
        # 4. Acceptance Rate (used to calculate M)
        # For psi(x)=2x, Integral(psi)=1.0. Acceptance Rate = 1.0 / L = 1.0 / 2.0 = 0.5
        if self.L <= 0:
             raise ParameterError("Bounding constant L must be positive.")
        # In a real package, C would be passed or approximated; here we assume C=1.0 for the example.
        self.expected_acceptance_rate = 1.0 / self.L 

    # --- Overridden Sampling Methods ---

    def __call__(self, n=None, n_min=None, n_max=None, return_weights=False, warn=True):
        """
        Directly calls gen_samples to apply the DAR filter. 
        Note: DAR samples inherently have unit weight (Jacobian=1) or weights are ignored, 
              as the filtering process handles the probability distortion.
        """
        if return_weights:
            # We enforce weights are not returned because the filtering process 
            # breaks the standard QMCPy Jacobian weight calculation.
            return self.gen_samples(n=n,n_min=n_min,n_max=n_max,warn=warn), np.ones(n)
        
        return self.gen_samples(n=n,n_min=n_min,n_max=n_max,warn=warn)

    def gen_samples(self, n=None, n_min=None, n_max=None, return_weights=False, warn=True):
        """
        Overrides the parent's method to implement the DAR sampling logic.
        """
        if n is None and n_max is not None and n_min is not None:
             N_target = n_max - n_min
        elif n is not None:
             N_target = n
        else:
             raise ParameterError("Must supply 'n' or 'n_min' and 'n_max'.")

        # --- Step 1A: Calculate Total Driver Points (M) ---
        M_required = math.ceil(N_target / self.expected_acceptance_rate)
        M_required = int(M_required)

        # --- Step 1B: Generate QMC Points (s=d+1 dimension) ---
        # We call the *driver* (s-dim) directly, passing M_required as 'n'
        Q_m_s = self.driver_distrib(n=M_required, warn=warn) 
        
        accepted_points = []
        
        # --- Step 2: Apply DAR Filter ---
        for x_full in Q_m_s:
            # 1. Driver Transformation: x = candidate (d dims), u = check variable (1 dim)
            x_candidate = x_full[:self.d] # d = self.d (target dimension)
            u_check = x_full[-1]          
            
            # 2. Evaluate Target Density (psi)
            psi_val = self.target_density(x_candidate)
            
            # 3. Acceptance Check: L * u <= psi(x)
            if self.L * u_check <= psi_val:
                accepted_points.append(x_candidate)
            
            # Stop when we hit the target N
            if len(accepted_points) >= N_target:
                break
        
        # --- Step 3: Return a clean sample array ---
        return np.array(accepted_points)

    # --- QMCPy Abstract Method Definitions ---
    # These are needed for the framework but are bypassed by our custom gen_samples/call logic.
    def _transform(self, x):
        """Transformation is handled internally by filtering in gen_samples."""
        raise MethodImplementationError(self, 'This TrueMeasure uses Acceptance-Rejection filtering and does not support _transform.')

    def _weight(self, x):
        """Weighting is handled implicitly by sampling density; weights are typically 1."""
        return np.ones(x.shape[0])
        
    def _spawn(self, sampler, dimension):
        """Spawning implementation for multi-level methods."""
        # For simplicity, we implement a direct copy for the sake of the exercise
        return AcceptanceRejectionMeasure(self.target_density, sampler, self.L)

# --- EXAMPLE TARGET DENSITY ---

def example_target_density(x_coord):
    """Density: psi(x_1, x_2) = 2*x_1 for x_i in [0,1]^2"""
    # x_coord is a 2D vector (d=2)
    return 2.0 * x_coord[0]

# --- EXECUTION EXAMPLE ---

TARGET_DIMENSION = 2 # d=2 (x and y)
N_TARGET = 1024      # N, the desired number of accepted samples
L_BOUND = 2.0        # L, the max value of psi(x,y)=2x
DRIVER_DIM = TARGET_DIMENSION + 1 # s=3

# 1. Initialize the QMC Driver (s=3)
driver_seq = MockSobol(dimension=DRIVER_DIM)

# 2. Initialize the DAR Measure (TrueMeasure)
dar_measure = AcceptanceRejectionMeasure(
    target_density=example_target_density,
    discrete_dist=driver_seq,
    L=L_BOUND
)

# 3. Generate the Samples
samples = dar_measure(n=N_TARGET)

# 4. Verification
mean_x_coord = np.mean(samples[:, 0])
print(f"\n--- Verification ---")
print(f"Target Dimension (d): {dar_measure.d}")
print(f"Driver Dimension (s): {dar_measure.driver_d}")
print(f"Shape of final accepted samples: {samples.shape}")
print(f"Mean of x_1 coordinate (Expected: 0.6667): {mean_x_coord:.4f}")


--- Verification ---
Target Dimension (d): 2
Driver Dimension (s): 3
Shape of final accepted samples: (1016, 2)
Mean of x_1 coordinate (Expected: 0.6667): 0.6757
